In [1]:
import zipfile
import os

base_path = "../"

train_zip = base_path + "train.zip"
test_zip = base_path + "test1.zip"

train_extract = base_path + "data/train"
test_extract = base_path + "data/test"

os.makedirs(train_extract, exist_ok=True)
os.makedirs(test_extract, exist_ok=True)

with zipfile.ZipFile(train_zip, 'r') as zip_ref:
    zip_ref.extractall(train_extract)

with zipfile.ZipFile(test_zip, 'r') as zip_ref:
    zip_ref.extractall(test_extract)

print("Extraction done")

Extraction done


In [2]:
import sys
!{sys.executable} -m pip install opencv-python scikit-image optuna scikit-learn matplotlib xgboost lightgbm

In [3]:
import cv2
import numpy as np
import os
from skimage.feature import hog

IMG_SIZE = 128  

def load_train_images(folder):
    x = []
    y = []
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        img = cv2.imread(path)
        if img is None:
            continue
        
        # Resize
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        # 🔥 Convert to grayscale (IMPORTANT for HOG)
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Normalize
        img_gray=img_gray/255.0
        
        # 🔥 HOG feature extraction
        features = hog(
            img_gray,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            feature_vector=True)
        
        x.append(features)
        # Labels
        if "cat" in file:
            y.append(0)
        else:
            y.append(1)
    
    return np.array(x), np.array(y)


train_path = "../data/train/train"

x, y = load_train_images(train_path)

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: (25000, 8100)
y shape: (25000,)


In [4]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

"""pca = PCA()
pca.fit(x)

import numpy as np
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Components")
plt.ylabel("Variance")
plt.show()"""

'pca = PCA()\npca.fit(x)\n\nimport numpy as np\nplt.plot(np.cumsum(pca.explained_variance_ratio_))\nplt.xlabel("Components")\nplt.ylabel("Variance")\nplt.show()'

In [5]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

xscaled=scaler.fit_transform(x)
pca=PCA(n_components=500)
X_pca=pca.fit_transform(xscaled)
var=np.cumsum(pca.explained_variance_ratio_)

components_95 = np.argmax(var >= 0.91) + 1
print(f"Number of components for 91% variance: {components_95}")

Number of components for 91% variance: 1


In [ ]:
"""cum=np.cumsum(pca.explained_variance_ratio_)
print(cum[100:300])"""


[0.80819068 0.80897951 0.80975973 0.81052881 0.81128429 0.81202681
 0.81276297 0.81349331 0.81421543 0.81492267 0.81562554 0.81632405
 0.81701568 0.81769508 0.81836839 0.81903105 0.81968513 0.82033485
 0.8209792  0.82161921 0.82225068 0.82288052 0.82350622 0.82412723
 0.82473421 0.82533762 0.82593385 0.82652332 0.8271084  0.82768633
 0.82825866 0.82882225 0.82938267 0.82994066 0.83049312 0.83104012
 0.83158492 0.83212477 0.8326537  0.83318084 0.8337004  0.83421883
 0.83473277 0.8352423  0.83574752 0.83624682 0.83674234 0.83723029
 0.83771582 0.83819484 0.83867113 0.8391431  0.83960568 0.84006754
 0.84052465 0.84097934 0.84143283 0.84187966 0.84232526 0.84276879
 0.84320739 0.84364433 0.84408011 0.84450878 0.8449357  0.84535526
 0.84577214 0.8461856  0.84659817 0.84700674 0.84741215 0.84781589
 0.84821658 0.84861119 0.84900193 0.84938867 0.84977299 0.85015298
 0.85053263 0.85090668 0.85127906 0.85164788 0.85201634 0.85238204
 0.85274441 0.85310492 0.85345967 0.8538102  0.85415993 0.8545

In [6]:
import joblib
joblib.dump(scaler,'scaler.pkl')
joblib.dump(pca,'pca.pkl')

['pca.pkl']

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import optuna

def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "use_label_encoder": False,
        "eval_metric": "logloss"
    }
    
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=20)

best_xgb = XGBClassifier(**study_xgb.best_params)
best_xgb.fit(X_train, y_train)

[I 2026-04-02 20:02:29,437] A new study created in memory with name: no-name-c5564949-84b7-4233-a2f1-595a48b0bbbd
c:\Users\HP\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:02:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-04-02 20:02:35,722] Trial 0 finished with value: 0.731 and parameters: {'n_estimators': 158, 'max_depth': 4, 'learning_rate': 0.2675958954205779, 'subsample': 0.9319766888704062, 'colsample_bytree': 0.9496413708089539, 'gamma': 1.938208872155408}. Best is trial 0 with value: 0.731.
c:\Users\HP\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:02:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-04-02 20:02:56,941] Trial 1 finished with value: 0.7204 and parameters: {'n_est

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7123544113641258
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [9]:
from lightgbm import LGBMClassifier

def objective_lgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", -1, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "class_weight":trial.suggest_categorical("class_weight",[None,"balanced"])
    }
    
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=20)

best_lgb = LGBMClassifier(**study_lgb.best_params)
best_lgb.fit(X_train, y_train)

[I 2026-04-02 20:09:31,343] A new study created in memory with name: no-name-0b1c0475-ad8e-4a83-a966-69226148b5dd


[LightGBM] [Info] Number of positive: 10000, number of negative: 10000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.055808 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-04-02 20:09:37,663] Trial 0 finished with value: 0.7394 and parameters: {'n_estimators': 242, 'max_depth': 8, 'learning_rate': 0.09566572704935555, 'num_leaves': 21, 'subsample': 0.6263538124594129, 'colsample_bytree': 0.6568903320877475, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7394.


[LightGBM] [Info] Number of positive: 10000, number of negative: 10000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.048512 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-04-02 20:09:38,685] Trial 1 finished with value: 0.6726 and parameters: {'n_estimators': 129, 'max_depth': 1, 'learning_rate': 0.03165140021995746, 'num_leaves': 86, 'subsample': 0.5054522879757759, 'colsample_bytree': 0.9630521224454823, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7394.


...

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-04-02 20:11:22,551] Trial 18 finished with value: 0.721 and parameters: {'n_estimators': 271, 'max_depth': 1, 'learning_rate': 0.17677581883059498, 'num_leaves': 97, 'subsample': 0.9999648594330398, 'colsample_bytree': 0.5807676020032442, 'class_weight': None}. Best is trial 10 with value: 0.7442.


[LightGBM] [Info] Number of positive: 10000, number of negative: 10000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036279 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-04-02 20:11:24,170] Trial 19 finished with value: 0.7356 and parameters: {'n_estimators': 221, 'max_depth': 3, 'learning_rate': 0.12217138148874912, 'num_leaves': 33, 'subsample': 0.8304881915072134, 'colsample_bytree': 0.5426206818627219, 'class_weight': 'balanced'}. Best is trial 10 with value: 0.7442.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.2875739787180269
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [10]:
from sklearn.svm import LinearSVC

best_svm = LinearSVC(max_iter=2500)
best_svm.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [11]:
models = {
    "SVM": best_svm,
    "XGB": best_xgb,
    "LGB": best_lgb
}

for name, model in models.items():
    print(name, model.score(X_val, y_val))

SVM 0.7332
XGB 0.7412
LGB 0.7442


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ("lgb", best_lgb),
        ("xgb", best_xgb),
        ("svm", best_svm)
    ],
    voting="hard"
)

voting.fit(X_train, y_train)
print("Voting:", voting.score(X_val, y_val))

[LightGBM] [Info] Number of positive: 10000, number of negative: 10000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033493 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Voting: 0.7496


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stack = StackingClassifier(
    estimators=[
        ("lgb", best_lgb),
        ("xgb", best_xgb),
        ("svm", best_svm)
    ],
    final_estimator=LogisticRegression(),
    cv=5
)

stack.fit(X_train, y_train)
print("Stack:", stack.score(X_val, y_val))

[LightGBM] [Info] Number of positive: 10000, number of negative: 10000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034380 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 8000, number of negative: 8000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027369 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8000, number of negative: 8000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030397 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8000, number of negative: 8000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029924 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


...

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8000, number of negative: 8000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027985 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Stack: 0.7512


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [16]:
vote=VotingClassifier(estimators=[
    ('lgb',best_lgb),
    ('xgb',best_xgb)
],
voting="soft",
weights=[2,1]
)

vote.fit(X_train, y_train)
joblib.dump(vote,"best_model.pkl")

[LightGBM] [Info] Number of positive: 10000, number of negative: 10000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030494 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127500
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 500
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


['best_model.pkl']